# TPM034A Machine Learning for socio-technical systems 
## `Lab session 01: Discover, explore and visualise data`

**Delft University of Technology**<br>
**Q2 2026**<br>
**Instructor:** Sander van Cranenburgh & Francisco Garrido Valenzuela<br>

### `Instructions`

**Lab session aim to:**<br>
* Show and reinforce how models and ideas presented in class are put to practice.<br>
* Help you gather hands-on machine learning skills.<br>

**Lab sessions are:**<br>
* Learning environments where you work with Jupyter notebooks and where you can get support from TAs and fellow students.<br> 
* Not graded and do not have to be submitted. 
<br><br><br>

### `Use of AI tools`

AI tools, such as ChatGPT and GitHub Copilot, are powerful tools to assist with programming. Moreover, in your future careers, you will work in a world where such tools are widely available. As such, we encourage you to learn to use AI tools effectively.

However, remember that the **primary objective of lab sessions is not to finish it, but to learn from it and develop your skills**. AI tools can help you achieve this, but they can also undermine your learning if you rely on them too heavily, see this recent [article](../../Assets/chatbots-and-thinking-english.pdf). Being able to instruct an AI tool to solve a problem is not the same as understanding how to solve it yourself. Understanding requires some effort and friction. You should therefore remain actively involved in conceptualising problems, structuring analyses, interpreting results, and critically evaluating AI-generated solutions.

> ⚠️ **Important:** We recommend using AI tools carefully, particularly because you will **NOT** have access to them during the *practical exam* (as internet access will be restricted). Therefore, at all times, make sure you understand what you are doing, not just that your code works.

### `Google Colab workspace set-up`

Uncomment the following cells code lines if you are running this notebook on Colab

In [ ]:
#!git clone https://github.com/TPM034A/TPM034A_2026
#!pip install -r TPM034A_2026/requirements.txt
#!mv "/content/TPM034A_2026/Lab_sessions/lab_session_01/data" /content/data

### `Application: liveability in the Netherlands` <br>
In this lab session, we investigate **liveability** across the Netherlands using data from the Dutch national liveability model (**Leefbaarometer 3.0**), developed for the Ministry of the Interior and Kingdom Relations (*Ministerie van Binnenlandse Zaken en Koninkrijksrelaties*).

The Leefbaarometer tracks liveability using 100+ objective indicators, synthesizing them into an overall liveability score (`lbm`) and five key sub-dimensions:
* **`won` (Housing):** Quality, value, type, and tenure of the housing stock.
* **`bew` / `soc` (Population / Social Cohesion):** Socio-economic status, demographic composition, and social stability.
* **`vrz` (Amenities / Facilities):** Proximity and accessibility to schools, supermarkets, general practitioners, and transit.
* **`fys` (Physical Environment):** Green space, air quality, noise exposure, and proximity to nuisance sources.
* **`vei` / `onv` (Safety / Nuisance):** Crime rates, recorded disturbances, and perceived nuisance.

In Leefbaarometer 3.0, the overall score (`lbm`) is expressed an absolute scale. The sub-dimensions (`won`, `soc`, `vrz`, `fys`, `onv`) are centered deviations from zero, where positive values represent above-average performance and negative values represent below-average performance.

**Learning objectives**. After completing the following exercises you will be able to: <br>
1. Load, explore, and enrich multi-level socio-technical datasets across administrative scales using `pandas`
2. Discover and visualise empirical distributions of composite indices and sub-dimensions using `seaborn`
3. Evaluate face validity and explore spatial disparities using `geopandas` and `folium`
4. Analyze bivariate associations and domain mechanisms using correlation heatmaps and municipal boxplots
5. Train and visually interpret `Decision Tree` regression models using `scikit-learn`
6. Investigate model complexity (`max_depth`), diagnose overfitting, and critically reflect on machine learning outputs for urban policy and decision-making

In [ ]:
# Import required Python packages and modules
import os
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import seaborn as sns
import matplotlib.pyplot as plt

# Scikit-learn modules for modeling, tree visualisation, and evaluation
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_squared_error, r2_score

# Configure plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 100

#### 1. Multi-level socio-technical data: loading and exploring

The data is aggregated at different spatial resolutions:

1. **`Gemeente` (Municipality):** Macro administrative level (e.g. Delft, Rotterdam, Amsterdam). High aggregation, suitable for national policy comparisons.
2. **`Wijk` (District):** Meso level representing coherent urban districts (e.g. *Binnenstad*, *Voorhof*).
3. **`Buurt` (Neighbourhood):** Micro level representing granular street blocks and residential pockets.

We provide pre-cleaned datasets for all three levels in the `data/` directory.

In [ ]:
# Set data directory path
data_folder = Path('data')
print(f"Data directory: {data_folder.resolve()}")

In [ ]:
# Load the cleaned datasets for all three spatial levels
gemeente_df = pd.read_csv(data_folder/'gemeente_data.csv')
wijk_df     = pd.read_csv(data_folder/'wijk_data.csv')
buurt_df    = pd.read_csv(data_folder/'buurt_data.csv')

In [ ]:
print(f"Gemeente level shape : {gemeente_df.shape} (Municipalities)")
print(f"Wijk level shape     : {wijk_df.shape} (Districts)")
print(f"Buurt level shape    : {buurt_df.shape} (Neighbourhoods)")

### `Variable descriptions`
The table below provides the full name and description for all key variables used across the datasets and exercises in this lab session:

| Variable | Term | Description |
| :--- | :--- | :--- |
| **Spatial Identifiers** | | |
| `GM_CODE` / `GM_NAAM` | *Gemeente (code / naam)* | Municipality identifier and official name (e.g. Delft, Rotterdam) |
| `WK_CODE` / `WK_NAAM` | *Wijk (code / naam)* | District identifier and official name (e.g. Binnenstad, Voorhof) |
| `BU_CODE` / `BU_NAAM` | *Buurt (code / naam)* | Neighbourhood identifier and official name (e.g. Centrum-Noord) |
| `Provincienaam` | *Provincienaam* | Province name merged from regional mapping (e.g. Zuid-Holland) |
| **Liveability Dimensions (Leefbaarometer 3.0)** | | |
| `lbm` | *Leefbaarometer score* | **Overall composite liveability score** (national baseline $\approx 4.0$) |
| `won` | *Woningvoorraad* | Housing sub-dimension: physical condition, building age, WOZ values |
| `soc` | *Bevolking / Sociaal* | Social cohesion sub-dimension: socio-economic status, stability, income |
| `vrz` | *Voorzieningen* | Amenities sub-dimension: proximity to shops, schools, healthcare |
| `fys` | *Fysieke omgeving* | Physical environment sub-dimension: green space, air quality, noise |
| `onv` | *Onveiligheid* | Safety/nuisance sub-dimension: crime rates, perceived safety, nuisance |
| **Demographics & Housing (CBS)** | | |
| `AANT_INW` | *Aantal inwoners* | Total number of registered residents |
| `BEV_DICHTH` | *Bevolkingsdichtheid* | Population density (inhabitants per $\text{km}^2$) |
| `P_HUURWON` | *Percentage huurwoningen* | Share of rental dwellings (social housing + private rental, %) |
| `P_KOOPWON` | *Percentage koopwoningen* | Share of owner-occupied private dwellings (%) |
| `P_EENP_HH` | *Eenpersoonshuishoudens* | Share of single-person households (%) |
| `P_65_EO_JR` | *65 jaar en ouder* | Share of senior residents aged 65 or older (%) |
| **Accessibility & Amenities (CBS)** | | |
| `AF_SUPERM` | *Afstand supermarkt* | Distance to the nearest supermarket (km) |
| `AF_ARTSPR` | *Afstand huisartsenpraktijk* | Distance to the nearest general practitioner (GP / doctor, km) |
| `AF_TREINST` | *Afstand treinstation* | Distance to the nearest railway passenger station (km) |

In [ ]:
# Preview top rows of the gemeente level data
gemeente_df[['GM_CODE', 'GM_NAAM', 'AANT_INW', 'BEV_DICHTH', 'lbm']].head()

### <span style="color:skyblue">Exercise 1: Exploring and enriching data with Pandas `.merge()`</span>
Often in socio-technical analysis, we need to enrich municipal data with regional administrative groupings (e.g. provinces).<br>
`A` Inspect the number of rows and columns in `gemeente_df`. How many municipalities are present in the Netherlands in this 2020 dataset?<br>
`B` Load the Excel mapping file `gemeente2provincie_mapping2020.xlsx` from the `data/` directory using `pd.read_excel()` and inspect its first few rows.<br>
`C` Perform a left merge using `pd.merge()` to add the province column (`Provincienaam`) to `gemeente_df` based on the matching municipality key `GM_CODE`. Verify that the new column was successfully added.

In [ ]:
# CODE YOUR ANSWERS HERE (Use as many cells as you need)

#### 2. Discovering and visualising data distributions
After loading socio-technical data, exploring the empirical distribution of key variables is essential.
Summary statistics (mean, standard deviation, skewness) and histograms reveal whether variables are normally distributed, bimodal, or skewed by extreme outliers.

In [ ]:
# Visualise the distribution of the mean and median of lbm at the wijk level
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=wijk_df, x='lbm', kde=True, bins=45, color='steelblue', ax=ax)
ax.axvline(wijk_df['lbm'].mean(), color='crimson', linestyle='--', linewidth=1.5, label=f"Mean: {wijk_df['lbm'].mean():.2f}")
ax.axvline(wijk_df['lbm'].median(), color='darkgreen', linestyle=':', linewidth=1.8, label=f"Median: {wijk_df['lbm'].median():.2f}")

ax.set_title("Distribution of Overall Liveability (lbm) at the District (Wijk) Level", fontsize=12, fontweight="bold")
ax.set_xlabel("Liveability Score (lbm)", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

**Interpretation of results:**
* Liveability at the district (*wijk*) level follows a unimodal, approximately bell-shaped (i.e. normal)distribution centered around $4.15$.
* The mean and median are closely aligned, showing a slight positive skew with an upper tail representing very liveable districts, and a lower tail representing poorly liveable districts.

### <span style="color:skyblue">Exercise 2: Distribution of social cohesion (`soc`)</span>
`A` Create a histogram with a Kernel Density Estimate (KDE) plot for the social cohesion sub-dimension (`soc`) at the district (*wijk*) level. See this documentation for more information about [kernel density estimation](https://seaborn.pydata.org/tutorial/distributions.html#kernel-density-estimation).<br>
`B` Compute and display its summary statistics: mean, standard deviation, minimum, maximum, median, and skewness (`.skew()`).<br>
`C` Compare the distribution of `soc` to `lbm`: Is `soc` centered around zero? What does a negative `soc` score signify in the Leefbaarometer framework?

In [ ]:
# CODE YOUR ANSWERS HERE (Use as many cells as you need)

#### 2.5 Evaluating face validity: Delft case study
An important step when discovering socio-technical data is to evaluate **face validity** (whether the statistical data aligns with real-world, intuitive domain knowledge of the system).

If an index indicates that a wealthy, leafy canal district has terrible liveability while an industrial docklands has high liveability, we would immediately question our data pipeline or indicator definitions!

Let's test face validity in our own university town: **Delft**!

Below, we demonstrate the complete discovery and spatial validation workflow for the **top 5 highest scoring neighbourhoods (*buurten*)** in Delft:
1. Filter the dataset for Delft and display the top 5 areas on liveability (`lbm`) alongside their sub-scores.
2. Load the neighbourhood boundary polygons and visualize them on an **interactive Leaflet web map** (using `geopandas.explore()`). You can zoom in, pan around, and hover over each neighbourhood to see its scores!

In [ ]:
# 1. Filter neighbourhoods for the municipality of Delft
delft_buurten = buurt_df[buurt_df['GM_NAAM'] == 'Delft'].copy()

# 2. Inspect top-5 highest scoring neighbourhoods in Delft
top_5_delft = delft_buurten.nlargest(5, 'lbm')[['BU_CODE', 'BU_NAAM', 'lbm', 'vrz', 'won', 'soc', 'fys', 'onv']]
print("Top 5 Highest-Scoring Neighbourhoods in Delft:")
display(top_5_delft)

# 3. Load geospatial boundaries for Delft
delft_map = gpd.read_file(data_folder/'delft_buurten.geojson')

# Identify top 5 neighbourhood codes
top_codes = top_5_delft['BU_CODE'].tolist()

# Create a categorization column for plotting
delft_map['top_highlight'] = 'Other neighbourhoods'
delft_map.loc[delft_map['BU_CODE'].isin(top_codes), 'top_highlight'] = 'Top 5 Liveability'

# 4. Interactive Leaflet map highlighting the Top 5
m_top = delft_map.explore(
    column='top_highlight',
    linecolor = 'black',
    linewidth = 2,
    cmap=['lightgrey', '#2ca02c'],
    categorical=True,
    tiles='OpenStreetMap',
    tooltip=['BU_NAAM', 'lbm', 'vrz', 'won', 'soc'],
    popup=True,
    legend=True,
    style_kwds={'fillOpacity': 0.7, 'weight': 1}
)
m_top

**Interpretation of results (Top 5):**
* The top 5 liveable neighbourhoods cluster in the historic city center (*Centrum-Zuidoost*, *Centrum-Oost*, *Centrum-West*, *In de Veste*) and the historic green garden village (*Agnetaparkbuurt*).
* Examining their sub-scores, these neighbourhoods boast exceptionally high amenity density (`vrz` around $+0.22$, reflecting walking proximity to grocery stores, restaurants, cultural institutions, and train stations) and strong residential attractiveness (`won`).
* This aligns closely with domain knowledge and lived experience in Delft, establishing strong **face validity** at the upper end of the index.

### <span style="color:skyblue">Exercise 3: Face validity of lower-scoring neighbourhoods in Delft</span>
Now that you have seen how to analyze and map the highest-scoring neighbourhoods, follow the exact same procedure for the **lowest-scoring neighbourhoods** in Delft:

`A` Find the 5 neighbourhoods (*buurten*) in Delft that score lowest on `lbm`. Display their `BU_CODE`, `BU_NAAM`, `lbm`, and sub-dimension scores (`won`, `soc`, `onv`, `vrz`, `fys`).<br>
`B` Create an interactive Leaflet map of Delft highlighting these **5 lowest-scoring neighbourhoods** in a contrasting color (e.g. crimson/red) with tooltips displaying their names and scores.<br>
`C` Reflect on their urban geography and socio-technical indicators: Where are these neighbourhoods located within Delft (e.g., *Voorhof*, *Buitenhof*)? Which sub-dimensions pull their liveability scores down? Does this spatial concentration align with your everyday experience or knowledge of Delft?

In [ ]:
# CODE YOUR ANSWERS HERE (Use as many cells as you need)

### <span style="color:skyblue">Exercise 4: Spatial disparities in liveability sub-dimensions</span> 
Liveability is a composite index. A neighbourhood may have good physical amenities but struggle with social cohesion or housing.<br>
`A` Create an interactive Leaflet map for Delft comparing two sub-dimensions: physical environment (`fys`) and social cohesion (`soc`). Add a `folium.LayerControl()` to toggle between both layers interactively.<br>
`B` Compare the two layers: Do neighbourhoods that score high in physical environment also score high in social cohesion, or do you observe spatial divergence?

In [ ]:
# CODE YOUR ANSWERS HERE (Use as many cells as you need)

# Load geospatial data for Delft
delft_geo = gpd.read_file(data_folder/'delft_buurten.geojson')

## Instructions for LayerControl()
# map1 = delft_geo.explore(...)
# delft_geo.explore(m = map1, ...)
# folium.LayerControl().add_to(m_sub)

#### 3. Exploring Associations: correlation heatmaps and municipal comparisons
To understand what structural factors relate to liveability, we analyze correlation matrices across demographic, housing, and environmental indicators, and compare liveability distributions across major Dutch municipalities.

In [ ]:
# Correlation matrix on neighbourhood (buurt) level
selected_vars = ['lbm', 'fys', 'onv', 'soc', 'vrz', 'won', 'P_HUURWON', 'P_KOOPWON', 'BEV_DICHTH']
corr_matrix = buurt_df[selected_vars].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlation Heatmap: Liveability Dimensions & Built Environment (Buurt Level)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### <span style="color:skyblue">Exercise 5: Interpreting bivariate correlations and domain intuition</span>
The heatmap displays both positive and negative bivariate correlations between liveability (`lbm`), its sub-dimensions, and built-environment indicators.<br>
`A` Select **one positive correlation** and **one negative correlation** from the heatmap involving liveability (`lbm`) or any of its sub-dimensions.<br>
`B` Reflect on whether these observed relationships align with your prior domain knowledge and intuition about urban liveability. What plausible socio-technical mechanisms explain why these indicators move together or in opposite directions?

In [ ]:
# CODE YOUR ANSWERS HERE (Use as many cells as you need)

**Boxplot Analysis across Major Municipalities** <br>
When evaluating urban systems, looking at national averages masks significant inter-city disparities. Below, we compare the liveability distributions across major Dutch cities (*G4* cities plus Delft, Eindhoven, and Groningen).

In [ ]:
# Boxplot comparing liveability across key municipalities
target_cities = ['Amsterdam', 'Rotterdam', "'s-Gravenhage", 'Utrecht', 'Delft', 'Eindhoven', 'Groningen']
city_buurten = buurt_df[buurt_df['GM_NAAM'].isin(target_cities)].copy()

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=city_buurten, x='GM_NAAM', y='lbm', order=target_cities, hue='GM_NAAM', palette='Set2', legend=False, ax=ax)
ax.axhline(4.0, color='red', linestyle='--', linewidth=1.5, label='National Baseline (4.0)')

ax.set_title("Liveability (lbm) Distribution across Major Dutch Municipalities", fontsize=13, fontweight="bold")
ax.set_xlabel("Municipality", fontsize=11)
ax.set_ylabel("Liveability Score (lbm)", fontsize=11)
ax.legend(loc='lower left', frameon=True)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Interpretation of results:**
* **Median differences across cities:** Amsterdam and Groningen feature the highest median liveability ($\approx 4.24$), whereas Delft and Rotterdam sit at the lower end ($\approx 4.08$), only slightly above the national baseline.
* **Spatial inequality within cities:** The Hague (*'s-Gravenhage*) and Amsterdam exhibit the highest internal dispersion (scores ranging widely from $3.6$ to $4.7$), highlighting acute spatial segregation between affluent and vulnerable neighbourhoods.
* **Neighbourhoods below national standard:** In Rotterdam, The Hague, and Delft, nearly 25% of neighbourhoods fall below the national baseline ($4.0$), demonstrating that national averages mask significant localized challenges in urban areas.

#### 4. Machine learning: predicting liveability with Decision Trees
In socio-technical policy analysis, linear models often struggle to capture **threshold effects** and **non-linear interactions** (e.g., an increase in rental housing may only depress liveability when density exceeds a critical threshold).

A **Decision Tree** is a non-parametric supervised learning algorithm that segments the feature space into rectangular regions using a sequence of binary if/then rules:
* **Root & Internal Nodes:** Test conditions on individual features (e.g., `P_KOOPWON <= 52.5%`).
* **Leaf Nodes:** The predicted liveability score (the average target value of training samples falling into that leaf).
* **Interpretability:** Unlike black-box models, a shallow decision tree can be plotted and visually inspected node-by-node!

Let's train a `DecisionTreeRegressor` on neighbourhood-level indicators to predict liveability (`lbm`).

In [ ]:
# Define feature matrix X and target vector y at the neighbourhood level
features = [
    'BEV_DICHTH',   # Population density (inhabitants/km2)
    'P_KOOPWON',    # % Owner-occupied housing
    'P_HUURWON',    # % Rental housing
    'P_EENP_HH',    # % Single-person households
    'P_65_EO_JR',   # % Elderly residents (65+)
    'AF_SUPERM',    # Distance to nearest supermarket (km)
    'AF_ARTSPR',    # Distance to nearest GP / doctor (km)
    'AF_TREINST',   # Distance to nearest train station (km)
]
target = 'lbm'

# Drop rows with missing values in the selected feature set
clean_buurt = buurt_df[features + [target]].dropna()

X = clean_buurt[features]
y = clean_buurt[target]

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=28)

print(f"Training samples: {X_train.shape[0]} neighbourhoods")
print(f"Test samples    : {X_test.shape[0]} neighbourhoods")

# Fit an interpretable baseline Decision Tree Regressor (max_depth=3)
tree_model = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)

# Evaluate predictions on both training and test sets
train_pred = tree_model.predict(X_train)
test_pred  = tree_model.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2  = r2_score(y_test, test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse  = np.sqrt(mean_squared_error(y_test, test_pred))

print(f"Train R²: {train_r2:.3f} | Train RMSE: {train_rmse:.3f}")
print(f"Test  R²: {test_r2:.3f} | Test  RMSE: {test_rmse:.3f}")

In [ ]:
# Visualise the trained Decision Tree structure
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    tree_model,
    feature_names=features,
    filled=True,
    rounded=True,
    precision=2,
    fontsize=9,
    ax=ax
)
ax.set_title("Trained Decision Tree for Neighbourhood Liveability (max_depth=3)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Interpretation of results:**
* **Root Split:** The decision tree places `P_HUURWON` at the root node. This confirms that housing tenure is the single strongest structural differentiator of neighbourhood liveability in the Netherlands.
* **Secondary Splits:** At the next level, the tree splits on population density (`BEV_DICHTH`) and elderly population share (`P_65_EO_JR`), distinguishing high-density urban flats from suburban/rural residential pockets.
* **Leaf Values:** Notice how the predicted liveability values vary from $\approx 3.82$ in high-density, high-rental leaves to $\approx 4.25$ in high-owner-occupied, lower-density leaves.
* **Interpretability:** Every single prediction can be justified with an explicit sequence of transparent socio-technical conditions.

### <span style="color:skyblue">Exercise 6: Investigating model complexity (tree depth & overfitting)</span>
In Decision Trees, the hyperparameter `max_depth` controls the depth of the tree, directly governing the tradeoff between **bias** (underfitting) and **variance** (overfitting).<br>
`A` Train a series of `DecisionTreeRegressor` models with `max_depth` varying from $1$ to $15$, as well as an unconstrained tree (`max_depth=None`).<br>
`B` For each depth, compute the $R^2$ score on both the training set and the test set, and store the results.<br>
`C` Plot the Train $R^2$ and Test $R^2$ curves across tree depths. At what depth does the model begin to overfit? What tree depth would you recommend to an urban policy department?

In [ ]:
# CODE YOUR ANSWERS HERE (Use as many cells as you need)

#### 5. Decision Tree Feature Importances

Decision trees provide an intrinsic measure of **feature importance** based on the total reduction in Mean Squared Error (variance reduction) brought by each feature across all splits.

Now we will extract the `feature_importances_` from a Decision Tree trained with `max_depth=6`,  and construct a sorted pandas Series, and plot them as a horizontal bar chart.


In [ ]:
best_tree = DecisionTreeRegressor(max_depth=6, random_state=42)
best_tree.fit(X_train, y_train)

importances = pd.Series(best_tree.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind='barh', color='teal', edgecolor='black', ax=ax)
ax.set_title("Feature Importances: Decision Tree Regressor (max_depth=6)", fontsize=12, fontweight="bold")
ax.set_xlabel("Gini / Variance Reduction Importance", fontsize=11)
ax.set_ylabel("Feature", fontsize=11)
plt.tight_layout()
plt.show()

### <span style="color:skyblue">Exercise 7: Policy Reflection</span>

Socio-Technical Policy Reflection: Write a 2-3 sentence reflection addressing:
* Which features carry the greatest explanatory weight in predicting liveability?
* Why might municipal policymakers and city council members prefer an interpretable Decision Tree with `max_depth=3` or `6` over a complex black-box model (like an ensemble or deep neural network), even if the black-box model achieves a slightly higher $R^2$?

In [ ]:
# CODE YOUR ANSWERS HERE (Use as many cells as you need)